# 02. Exploración y limpieza inicial de los datos

**Puntos de la rúbrica:** b (EDA: descriptiva, tipos de datos, tendencia central) y c (análisis preliminar y necesidad de información adicional).

## Para qué sirve este notebook

Es la primera mirada a los datos, como la haría alguien que **no conoce la base**: antes de hacer gráficas hay que entender **qué hay, qué falta y qué hay que arreglar**. De esa exploración sale, más adelante, la pregunta analítica.

## Plan

1. **Las dos tablas:** validaciones (el consolidado del notebook 01) y estaciones (dos archivos).
2. **Estructura de las validaciones:** columnas, tipos de dato y qué significa cada una.
3. **Valores faltantes** por columna.
4. **Columnas que no aportan:** constantes y vacías.
5. **Fechas:** rango, días completos y "colas de medianoche".
6. **Variables categóricas:** qué valores hay y con qué frecuencia.
7. **Variables numéricas:** medidas de tendencia central y dispersión.
8. **Estaciones:** calidad y cruce con las validaciones.
9. **Resumen de limpieza:** qué problema hay, a cuánto afecta y qué se decide.

> Se avanza **un paso a la vez**: cada sección se revisa antes de pasar a la siguiente. La limpieza no borra nada del consolidado: se documentan las decisiones y se aplican al crear las tablas de análisis.

## 1. Las dos tablas

Hay **dos fuentes** que se van a cruzar:

| Tabla | De dónde viene | Qué aporta |
|---|---|---|
| **Validaciones** | El consolidado Parquet del notebook 01 (31.5 millones de filas) | Cada fila es **una pasada de tarjeta** por un validador de una estación |
| **Estaciones** | Dos archivos CSV de datos abiertos (`estaciones_troncales.csv` y `estaciones_troncales_ui.csv`) | Datos de cada estación: nombre, troncal, accesos, vagones y **coordenadas** (las validaciones no traen latitud ni longitud) |

Las validaciones se abren con **Dask** (perezoso: no se carga todo en memoria); las estaciones, que son pequeñas, con pandas.

In [1]:
# QUÉ HACE: abre las dos fuentes: el consolidado de validaciones (con Dask) y los dos archivos de estaciones (con pandas).
# CÓMO: dd.read_parquet es perezoso (solo lee metadatos, por eso es instantáneo); los CSV de estaciones se leen completos.
# QUÉ MIRAR: validaciones 31.5 millones x 22 columnas; el archivo de estaciones 1 tiene ~149 filas y el 2 unas 153 con columnas distintas.
import numpy as np
import pandas as pd
import dask.dataframe as dd
from pathlib import Path

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

VALIDACIONES = Path('../data/processed/validaciones')
ESTACIONES = Path('../data/raw/estaciones')

val = dd.read_parquet(str(VALIDACIONES))
est_1 = pd.read_csv(ESTACIONES / 'estaciones_troncales.csv', encoding='utf-8-sig')
est_2 = pd.read_csv(ESTACIONES / 'estaciones_troncales_ui.csv', encoding='utf-8-sig')

print(f"Validaciones:                    {len(val):,} filas x {len(val.columns)} columnas ({val.npartitions} particiones)")
print(f"Estaciones, archivo 1 (API):     {est_1.shape[0]} filas x {est_1.shape[1]} columnas")
print(f"Estaciones, archivo 2 (descarga): {est_2.shape[0]} filas x {est_2.shape[1]} columnas")

Validaciones:                    31,530,646 filas x 22 columnas (21 particiones)
Estaciones, archivo 1 (API):     150 filas x 27 columnas
Estaciones, archivo 2 (descarga): 153 filas x 29 columnas


In [2]:
# QUÉ HACE: muestra las primeras filas de cada tabla (las columnas se ven en vertical para poder leerlas todas).
# CÓMO: .head(3) de cada tabla; .T las transpone (columnas en filas) porque son muchas.
# QUÉ MIRAR: en validaciones, una fila = una pasada de tarjeta (estación, fecha-hora, perfil, saldos); en estaciones, nombre, troncal y coordenadas.
print('VALIDACIONES (3 primeras filas)')
display(val.head(3).T)
print('\nESTACIONES, archivo 1 (3 primeras filas)')
display(est_1.head(3).T)
print('\nESTACIONES, archivo 2 (3 primeras filas)')
display(est_2.head(3).T)

VALIDACIONES (3 primeras filas)


,0,1,2
Acceso_Estacion,(17) Acceso Peatonal,(17) Acceso Peatonal,(03) Acceso Sur
Day_Group_Type,Dia 1,Dia 1,Dia 1
Dispositivo,40000202,40000204,40000310
Emisor,(3101000) Bogota Card(Citizen),(3101000) Bogota Card(Citizen),(3101000) Bogota Card(Citizen)
Estacion_Parada,(08000) Portal Tunal,(08000) Portal Tunal,(05002) Tintal
Fase,Fase 3,Fase 3,Fase 3
Fecha_Clearing,2026-08-03,2026-08-03,2026-08-03
Fecha_Transaccion,2026-08-03 03:44:41,2026-08-03 03:44:58,2026-08-03 03:45:49
Hora_Pico_SN,Peak Time,Peak Time,Peak Time
ID_Vehiculo,NaN,NaN,NaN



ESTACIONES, archivo 1 (3 primeras filas)


,0,1,2
X,997836.0595,998139.4242,995130.6531
Y,996629.2544,1016655.2375,1009444.2454
objectid,1,2,3
numero_estacion,10000,3000,6000
nombre_estacion,Portal 20 de Julio,Portal Suba,Portal El Dorado – C.C. NUESTRO BOGOTÁ
coordenada_x_estacion,997836.0595,998139.4242,995130.6531
coordenada_y_estacion,996629.2544,1016655.2375,1009444.2454
ubicacion_estacion,Kra 5 a Cl 33 Sur,Av Suba Av C. Cali,KR 87
troncal_estacion,Cr 7-10,Suba,Calle 26
numero_vagones_estacion,1,1,1



ESTACIONES, archivo 2 (3 primeras filas)


,0,1,2
X,1000325.4654,997610.17,999866.6112
Y,1007757.54,1005603.8612,1005440.2936
observ,Valores de capacidad de buses en proceso de ac...,Valores de capacidad de buses en proceso de ac...,Valores de capacidad de buses en proceso de ac...
longitud,-74.074575,-74.099046,-74.07871
latitud,4.666353,4.646876,4.645398
objectid,624,625,626
num_est,7103,6103,7106
nom_est,AV. Chile,CAN,Campín - UAN
id_trazado,TZ008,TZ016,TZ008
long_est,173.64,96.48,122.82


## 2. Estructura de las validaciones

La tabla de validaciones tiene **22 columnas**. Para que cualquiera la entienda, se arma una tabla-diccionario con, para cada columna: su **tipo de dato**, un **ejemplo** real y **qué significa**. Las columnas se agrupan por la pregunta que responden: *cuándo, dónde, quién, cuánto*, y dos grupos que ya se ven sospechosos (columnas con un único valor y columnas vacías).

In [3]:
# QUÉ HACE: arma el diccionario de datos: una fila por columna con su grupo, tipo, un ejemplo y qué significa.
# CÓMO: el tipo sale de la tabla real (val.dtypes), el ejemplo de la primera fila y la descripción de un diccionario escrito a mano.
# QUÉ MIRAR: cuántas columnas son útiles (cuándo, dónde, quién, cuánto) y cuántas parecen no aportar (un solo valor o vacías: se confirma en las secciones 3 y 4).
pd.set_option('display.max_colwidth', 90)

DESCRIPCION = {
    'Fecha_Transaccion': ('1. Cuándo', 'Fecha y hora exactas de la validación'),
    'Fecha_Clearing': ('1. Cuándo', 'Fecha de compensación contable del recaudo'),
    'Estacion_Parada': ('2. Dónde', 'Estación: código de 5 dígitos y nombre juntos, entre paréntesis y texto'),
    'Acceso_Estacion': ('2. Dónde', 'Acceso o puerta de la estación por donde entró la persona'),
    'Linea': ('2. Dónde', 'Línea o zona (troncal) a la que pertenece la estación'),
    'Dispositivo': ('2. Dónde', 'Código del validador (equipo) que leyó la tarjeta'),
    'Numero_Tarjeta': ('3. Quién', 'Número de tarjeta anonimizado (identifica la tarjeta, no a la persona)'),
    'Nombre_Perfil': ('3. Quién', 'Perfil del usuario de la tarjeta (adulto, adulto mayor, anónimo, etc.)'),
    'Tipo_Tarjeta': ('3. Quién', 'Tipo de tarjeta tuLlave (Plus o Básica)'),
    'Emisor': ('3. Quién', 'Entidad que emitió el medio de pago (tuLlave o bancos)'),
    'Valor': ('4. Cuánto', 'Valor de la transacción'),
    'Tipo_Tarifa': ('4. Cuánto', 'Código del tipo de tarifa aplicada (su significado se explora más adelante)'),
    'Saldo_Previo_a_Transaccion': ('4. Cuánto', 'Saldo de la tarjeta antes de validar'),
    'Saldo_Despues_Transaccion': ('4. Cuánto', 'Saldo de la tarjeta después de validar'),
    'Day_Group_Type': ('5. Sistema (¿un solo valor?)', 'Tipo de día según el sistema de recaudo'),
    'Hora_Pico_SN': ('5. Sistema (¿un solo valor?)', 'Indicador de hora pico según el sistema de recaudo'),
    'Fase': ('5. Sistema (¿un solo valor?)', 'Fase del sistema de transporte'),
    'Operador': ('5. Sistema (¿un solo valor?)', 'Operador del servicio'),
    'Sistema': ('5. Sistema (¿un solo valor?)', 'Sistema de transporte (troncal)'),
    'ID_Vehiculo': ('6. Vehículo (¿vacías?)', 'Identificador del vehículo'),
    'Ruta': ('6. Vehículo (¿vacías?)', 'Ruta del bus'),
    'Tipo_Vehiculo': ('6. Vehículo (¿vacías?)', 'Tipo de vehículo'),
}

primera_fila = val.head(1).iloc[0]
estructura = pd.DataFrame({
    'grupo': pd.Series({c: DESCRIPCION[c][0] for c in val.columns}),
    'tipo': val.dtypes.astype(str),
    'ejemplo': primera_fila.astype(str).str.slice(0, 40),
    'qué_es': pd.Series({c: DESCRIPCION[c][1] for c in val.columns}),
})
estructura = estructura.sort_values('grupo', kind='stable')
estructura.index.name = 'columna'
estructura

,grupo,tipo,ejemplo,qué_es
columna,,,,
Fecha_Clearing,1. Cuándo,string,2026-08-03,Fecha de compensación contable del recaudo
Fecha_Transaccion,1. Cuándo,string,2026-08-03 03:44:41,Fecha y hora exactas de la validación
Acceso_Estacion,2. Dónde,string,(17) Acceso Peatonal,Acceso o puerta de la estación por donde entró la persona
Dispositivo,2. Dónde,int64,40000202,Código del validador (equipo) que leyó la tarjeta
Estacion_Parada,2. Dónde,string,(08000) Portal Tunal,"Estación: código de 5 dígitos y nombre juntos, entre paréntesis y texto"
Linea,2. Dónde,string,(34) Zona H Caracas Sur,Línea o zona (troncal) a la que pertenece la estación
Emisor,3. Quién,string,(3101000) Bogota Card(Citizen),Entidad que emitió el medio de pago (tuLlave o bancos)
Nombre_Perfil,3. Quién,string,(001) Adulto,"Perfil del usuario de la tarjeta (adulto, adulto mayor, anónimo, etc.)"
Numero_Tarjeta,3. Quién,string,113afe3135a92083c79c0095fde0a2fc78b30dfd,"Número de tarjeta anonimizado (identifica la tarjeta, no a la persona)"


## 3. Valores faltantes por columna

Primera revisión de calidad: **¿cuántos datos faltan en cada columna?** Se cuentan los vacíos (`NaN`) sobre las **31.5 millones de filas completas** (no sobre una muestra), con Dask.

In [4]:
# QUÉ HACE: cuenta los valores faltantes de cada columna en las 31.5 millones de filas.
# CÓMO: val.isna().sum() lo calcula Dask recorriendo todo el consolidado (con 3 hilos a la vez, para cuidar la memoria).
# QUÉ MIRAR: columnas 100 % vacías (no sirven); columnas con muy pocos vacíos (se pueden tratar); el resto, completas.
import time
import dask

dask.config.set(scheduler='threads', num_workers=3)
N = len(val)                                   # total de filas

t0 = time.perf_counter()
vacios = val.isna().sum().compute()
print(f"Conteo de vacíos en {N:,} filas: {time.perf_counter() - t0:.0f} s")

faltantes = pd.DataFrame({
    'tipo': val.dtypes.astype(str),
    'vacíos': vacios,
    'vacíos_%': (vacios / N * 100).round(3),
})
faltantes['estado'] = np.where(faltantes['vacíos_%'] == 100, 'VACÍA',
                       np.where(faltantes['vacíos'] > 0, 'con algunos vacíos', 'completa'))
faltantes = faltantes.sort_values('vacíos_%', ascending=False)
faltantes.index.name = 'columna'
faltantes

Conteo de vacíos en 31,530,646 filas: 7 s


,tipo,vacíos,vacíos_%,estado
columna,,,,
Ruta,float64,31530646,100.000,VACÍA
ID_Vehiculo,float64,31530646,100.000,VACÍA
Tipo_Vehiculo,float64,31530646,100.000,VACÍA
Linea,string,2438,0.008,con algunos vacíos
Acceso_Estacion,string,0,0.000,completa
Day_Group_Type,string,0,0.000,completa
Fase,string,0,0.000,completa
Estacion_Parada,string,0,0.000,completa
Emisor,string,0,0.000,completa


**Lectura (sección 3, valores faltantes)**

- La base está **muy completa**: 18 de las 22 columnas no tienen ni un solo vacío.
- **3 columnas están 100 % vacías** (`ID_Vehiculo`, `Ruta`, `Tipo_Vehiculo`): son datos de vehículos que no aplican a la troncal.
- `Linea` tiene **2,438 vacíos (0.008 %)**. Es el mismo número de validaciones que tienen el acceso `(00)` (sección 6), así que casi seguro son registros de un acceso sin definir.
- No hay que rellenar ni inventar nada: son muy pocos y se pueden dejar como "sin línea".

## 4. Columnas que no aportan: constantes y vacías

Una columna **no sirve para analizar** si está vacía o si todas las filas tienen **el mismo valor** (no distingue nada entre una validación y otra). Se cuentan los valores distintos de cada columna y se anota qué se hace con ella.

In [5]:
# QUÉ HACE: cuenta los valores distintos de cada columna y decide si la columna se usa o no.
# CÓMO: conteo exacto en las columnas con hasta 5,000 valores distintos; en las que tienen millones (fecha-hora, tarjeta, saldos) un conteo aproximado (error de ~1-2 %).
# QUÉ MIRAR: columnas con 1 solo valor (constantes) o 0 (vacías): no sirven para analizar; Day_Group_Type tiene 2 valores (se explica más abajo).
aprox = dask.compute({c: val[c].nunique_approx() for c in val.columns})[0]

distintos, valor_unico, tipo_conteo = {}, {}, {}
for c in val.columns:
    if aprox[c] <= 5000:                                # pocas categorías: conteo exacto
        valores = val[c].drop_duplicates().dropna().compute()
        distintos[c], tipo_conteo[c] = len(valores), 'exacto'
        valor_unico[c] = valores.iloc[0] if len(valores) == 1 else ''
    else:
        distintos[c], tipo_conteo[c], valor_unico[c] = int(round(aprox[c])), 'aproximado', ''

columnas = pd.DataFrame({'valores_distintos': distintos, 'conteo': tipo_conteo, 'valor_único': valor_unico,
                         'vacíos_%': faltantes['vacíos_%']})
columnas['decisión'] = np.where(columnas['vacíos_%'] == 100, 'No se usa: vacía',
                         np.where(columnas['valores_distintos'] == 1, 'No se usa: siempre el mismo valor', 'Se usa'))
columnas.index.name = 'columna'
columnas.sort_values(['decisión', 'valores_distintos'])

,valores_distintos,conteo,valor_único,vacíos_%,decisión
columna,,,,,
Fase,1,exacto,Fase 3,0.000,No se usa: siempre el mismo valor
Hora_Pico_SN,1,exacto,Peak Time,0.000,No se usa: siempre el mismo valor
Operador,1,exacto,(201) Trunk agency,0.000,No se usa: siempre el mismo valor
Sistema,1,exacto,TRONCAL,0.000,No se usa: siempre el mismo valor
ID_Vehiculo,0,exacto,,100.000,No se usa: vacía
Ruta,0,exacto,,100.000,No se usa: vacía
Tipo_Vehiculo,0,exacto,,100.000,No se usa: vacía
Day_Group_Type,2,exacto,,0.000,Se usa
Tipo_Tarjeta,2,exacto,,0.000,Se usa


**Lectura (sección 4, columnas que no aportan)**

- **7 de las 22 columnas no sirven para analizar:** 3 vacías y 4 con un solo valor (`Fase` = "Fase 3", `Hora_Pico_SN` = "Peak Time", `Operador` = "(201) Trunk agency", `Sistema` = "TRONCAL"). Quedan **15 columnas útiles**.
- **`Hora_Pico_SN` siempre dice "Peak Time"**, o sea que no indica qué es hora pico: la hora pico hay que definirla nosotros a partir de la hora.
- **`Day_Group_Type` NO es constante** (aunque en un solo día lo parecía): vale `Dia 1` casi siempre y `Dia 2` solo en los **domingos** (9 y 16 de agosto, 20 de septiembre). **No distingue los sábados ni el festivo** (el 7 de agosto también es `Dia 1`), por eso el tipo de día se construye desde la fecha.
- Columnas con **millones de valores distintos**: `Numero_Tarjeta` (≈ 3.9 millones de tarjetas) y `Fecha_Transaccion` (≈ 1.4 millones de segundos distintos). Estos conteos son aproximados; los demás son exactos.
- Estas columnas **no se borran del consolidado**: simplemente no se usan al analizar.

## 5. Fechas: rango, días completos y "colas de medianoche"

La fecha-hora de cada validación está guardada como **texto** (`2026-08-03 03:44:41`). Se revisa: ¿qué rango de fechas cubre?, ¿todos los días están completos?, ¿hay validaciones sueltas de días que no deberían estar? Además se compara con `Fecha_Clearing`, la fecha de compensación contable, que es la que usa el nombre de cada archivo.

In [6]:
# QUÉ HACE: cuenta las validaciones de cada fecha y marca cuáles días están completos y cuáles solo traen unas pocas filas.
# CÓMO: se toman los 10 primeros caracteres de Fecha_Transaccion (AAAA-MM-DD) y se cuentan; un día con menos de 10,000 validaciones se marca "cola de medianoche".
# QUÉ MIRAR: 21 días completos más unas pocas fechas "sueltas" (17-ago, 13-sep, 21-sep); los domingos y el festivo (7-ago) tienen mucha menos demanda.
DIAS = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']

largo = val['Fecha_Transaccion'].str.len().value_counts().compute()
print(f"Largo del texto de la fecha-hora: {largo.to_dict()}  (un solo largo = formato uniforme)")

por_fecha = (val['Fecha_Transaccion'].str[:10].value_counts().compute().sort_index()
             .rename('validaciones').rename_axis('fecha').to_frame())
por_fecha['día'] = [DIAS[d.dayofweek] for d in pd.to_datetime(por_fecha.index)]
por_fecha['estado'] = np.where(por_fecha['validaciones'] < 10_000, 'cola de medianoche (incompleto)', 'día completo')
print(f"Rango: {por_fecha.index.min()} a {por_fecha.index.max()} | fechas distintas: {len(por_fecha)}")
por_fecha

Largo del texto de la fecha-hora: {np.int64(19): 31530646}  (un solo largo = formato uniforme)


Rango: 2026-08-03 a 2026-09-21 | fechas distintas: 24


,validaciones,día,estado
fecha,,,
2026-08-03,1817358,lunes,día completo
2026-08-04,1879861,martes,día completo
2026-08-05,1893706,miércoles,día completo
2026-08-06,1837051,jueves,día completo
2026-08-07,525834,viernes,día completo
2026-08-08,1062203,sábado,día completo
2026-08-09,516447,domingo,día completo
2026-08-10,1751957,lunes,día completo
2026-08-11,1832549,martes,día completo


In [7]:
# QUÉ HACE: compara la fecha real de la validación con Fecha_Clearing (la fecha del archivo) y cuenta cuántas filas no coinciden.
# CÓMO: agrupa por (Fecha_Clearing, fecha de la validación); diferencia_días = Fecha_Clearing - fecha de la validación.
# QUÉ MIRAR: casi todas las filas con diferencia 0; las pocas filas con diferencia distinta de 0 son validaciones que quedaron en el archivo de otro día.
cruce = (val.assign(fecha_validacion=val['Fecha_Transaccion'].str[:10])
            .groupby(['Fecha_Clearing', 'fecha_validacion']).size().compute()
            .rename('validaciones').reset_index())
cruce['diferencia_días'] = (pd.to_datetime(cruce['Fecha_Clearing']) - pd.to_datetime(cruce['fecha_validacion'])).dt.days

diferencias = cruce.groupby('diferencia_días')['validaciones'].sum().to_frame()
diferencias['%'] = (diferencias['validaciones'] / N * 100).round(4)
print(f"Fechas de Fecha_Clearing distintas: {cruce['Fecha_Clearing'].nunique()} (una por archivo)")
display(diferencias)
print('Filas cuya fecha de validación NO es la del archivo:')
cruce[cruce['diferencia_días'] != 0].sort_values('validaciones', ascending=False).head(12)

Fechas de Fecha_Clearing distintas: 21 (una por archivo)


,validaciones,%
diferencia_días,,
-1,240,0.0008
0,31527816,99.9910
1,2590,0.0082


Filas cuya fecha de validación NO es la del archivo:


,Fecha_Clearing,fecha_validacion,validaciones,diferencia_días
26,2026-08-16,2026-08-15,2014,1
29,2026-09-14,2026-09-13,314,1
32,2026-09-15,2026-09-14,262,1
19,2026-08-12,2026-08-13,36,-1
23,2026-08-14,2026-08-15,22,-1
42,2026-09-19,2026-09-20,22,-1
40,2026-09-18,2026-09-19,21,-1
7,2026-08-06,2026-08-07,19,-1
36,2026-09-16,2026-09-17,18,-1
25,2026-08-15,2026-08-16,16,-1


**Lectura (sección 5, fechas)**

- Hay **24 fechas**: **21 días completos** (3 al 16 de agosto y 14 al 20 de septiembre) y **3 "colas de medianoche"** con muy pocas filas (17-ago: 1 validación; 13-sep: 314; 21-sep: 10).
- El formato de la fecha-hora es **uniforme** (19 caracteres en todas las filas), así que se puede convertir a fecha sin problemas.
- La demanda cambia mucho según el día: los **días hábiles** mueven entre 1.75 y 1.92 millones de validaciones, los **sábados** entre 1.06 y 1.23 millones y los **domingos** entre 0.52 y 0.59 millones. El **festivo (7-ago)** tiene 525,834: se parece a un domingo, no a un día hábil.
- `Fecha_Clearing` (la fecha del archivo) coincide con la fecha real de la validación en el **99.99 %** de las filas. Solo **2,830 filas (0.009 %)** quedaron en el archivo de otro día; casi todas son validaciones de la noche anterior (por ejemplo, 2,014 del 15-ago en el archivo del 16-ago).
- **Decisión:** el análisis usa `Fecha_Transaccion` (la fecha real), no la del archivo, y las tres colas de medianoche se excluyen de los promedios por tipo de día.

## 6. Variables categóricas: qué valores hay y con qué frecuencia

Las categóricas son columnas con **pocos valores posibles** (perfil, tipo de tarjeta, tipo de tarifa, emisor, línea...). Para cada una se muestra una tabla con cada valor, cuántas validaciones tiene y qué porcentaje del total es. En las que tienen muchísimos valores (estaciones, accesos) se muestran solo los 10 más frecuentes. En una variable categórica la **moda** (el valor más frecuente) es la medida de tendencia central.

In [8]:
# QUÉ HACE: muestra, para cada variable categórica, cuántas validaciones tiene cada valor y qué % del total representa.
# CÓMO: value_counts() de Dask sobre toda la columna; en estaciones y accesos solo se muestran los 10 más frecuentes (el resto es muy largo).
# QUÉ MIRAR: la moda de cada columna (primera fila de cada tabla, la de más validaciones); qué tan repartidos o concentrados están los valores; cuántas validaciones son tarjeta anónima (Básica).
def frecuencias(columna):
    conteo = val[columna].value_counts().compute().sort_values(ascending=False)
    tabla = pd.DataFrame({'validaciones': conteo, '%': (conteo / N * 100).round(2)})
    tabla.index.name = columna
    return tabla

for col in ['Day_Group_Type', 'Nombre_Perfil', 'Tipo_Tarjeta', 'Tipo_Tarifa', 'Emisor', 'Linea']:
    tabla = frecuencias(col)
    print(f"\n=== {col} ({len(tabla)} valores distintos) ===")
    display(tabla)

for col in ['Estacion_Parada', 'Acceso_Estacion']:
    tabla = frecuencias(col)
    print(f"\n=== {col} ({len(tabla)} valores distintos): los 10 más frecuentes ===")
    display(tabla.head(10))


=== Day_Group_Type (2 valores distintos) ===


,validaciones,%
Day_Group_Type,,
Dia 1,29884244,94.78
Dia 2,1646402,5.22



=== Nombre_Perfil (7 valores distintos) ===


,validaciones,%
Nombre_Perfil,,
(001) Adulto,15908803,50.46
(001) Anonymous,9412307,29.85
(002) Adulto Mayor,2599663,8.24
(006) Apoyo Ciudadano,1225084,3.89
(009) Apoyo Ciudadano Reexpedición,946308,3.0
(101) Adulto PV,927128,2.94
(005) Discapacidad,511353,1.62



=== Tipo_Tarjeta (2 valores distintos) ===


...

,validaciones,%
Linea,,
(33) Zona B AutoNorte,5210209,16.52
(30) Zona G NQS Sur,4328560,13.73
(34) Zona H Caracas Sur,3240031,10.28
(31) Zona F Av. Américas,2971883,9.43
(11) Zona K Calle 26,2940533,9.33
(12) Zona L Carrera 10,2680807,8.5
(36) Zona A Caracas,2551439,8.09
(32) Zona C Av. Suba,2366949,7.51
(35) Zona D Calle 80,2192849,6.95



=== Estacion_Parada (155 valores distintos): los 10 más frecuentes ===


,validaciones,%
Estacion_Parada,,
(02000) Portal Norte - Unicervantes,1246890,3.95
(05000) Portal Américas,1241261,3.94
(03000) Portal Suba,985417,3.13
(07000) Portal Sur JFK Coop. Financiera,984323,3.12
(04000) Cabecera Calle 80,927639,2.94
(09000) Cabecera Usme,899454,2.85
(06000) Portal El Dorado - C.C. NUESTRO BOGOTA,829735,2.63
(10000) Portal 20 de Julio,749214,2.38
(08000) Portal Tunal,728652,2.31



=== Acceso_Estacion (433 valores distintos): los 10 más frecuentes ===


,validaciones,%
Acceso_Estacion,,
(01) Acceso Norte,1858068,5.89
(02) Acceso Sur,1184758,3.76
(01) Acceso Peatonal Oriental,969080,3.07
(01) Acceso Peatonal Sur,940137,2.98
(03) Acceso Norte,870570,2.76
(04) Acceso Sur,835085,2.65
(01) Acceso Peatonal Norte,774394,2.46
(AA) Acceso Norte,729632,2.31
(01) Acceso Sur,534520,1.7


**Lectura (sección 6, categóricas)**

- **Tipo de usuario (`Nombre_Perfil`):** la moda es **Adulto** (50.5 %). Le siguen la tarjeta **anónima** (29.9 %), el **adulto mayor** (8.2 %), el **apoyo ciudadano** (6.9 % sumando sus dos códigos), **adulto PV** (2.9 %) y **discapacidad** (1.6 %).
- **`Tipo_Tarjeta` es redundante con el perfil:** "tullave Básica" tiene exactamente las mismas 9,412,307 validaciones que el perfil "Anonymous". O sea que casi **1 de cada 3 validaciones es de tarjeta anónima** (Básica) y las otras son tuLlave Plus.
- **`Tipo_Tarifa`:** el código `1,0` es el 87.1 % de las validaciones; `2,0` el 12.5 % y `3,0` el 0.4 %. Como el valor cobrado es 0 en el 13.9 % de las filas (sección 7), parece que los códigos 2 y 3 son distintos del pasaje normal. **Qué significan hay que explorarlo más adelante.**
- **`Emisor`:** el 95.9 % de las validaciones usa tarjeta tuLlave; el resto son tarjetas bancarias y otros medios de pago.
- **`Linea`:** 13 troncales (más las filas sin línea); las más cargadas son **AutoNorte (16.5 %)** y **NQS Sur (13.7 %)**.
- **Estaciones y accesos:** hay **154 códigos de estación** y 433 accesos distintos. Solo se muestran los 10 más frecuentes; la comparación entre estaciones se hace en el siguiente notebook.

## 7. Variables numéricas: tendencia central y dispersión

Las columnas numéricas con significado son `Valor` y los dos saldos (`Dispositivo` es un código de equipo, no una cantidad). Para cada una se calculan las medidas básicas:

| Medida | Qué dice |
|---|---|
| **Media** | el promedio |
| **Mediana** (50 %) | el valor del medio: la mitad de las filas está por debajo |
| **Moda** | el valor más repetido |
| **Desviación estándar** | qué tan dispersos están los datos alrededor de la media |
| **Mínimo, cuartiles y máximo** | el rango y cómo se reparten los datos |
| **Asimetría** | si la cola está hacia valores altos (positiva) o bajos (negativa) |

Después se hacen unas **verificaciones de consistencia** (¿hay valores negativos o imposibles?).

In [9]:
# QUÉ HACE: calcula las medidas de tendencia central y de dispersión de Valor y de los dos saldos.
# CÓMO: describe() de Dask (los percentiles son aproximados), más moda (value_counts) y asimetría (skew).
# QUÉ MIRAR: media y mediana de Valor (¿se parecen?); la moda; saldos con media muy distinta de la mediana y asimetría alta = pocos valores muy grandes.
numericas = ['Valor', 'Saldo_Previo_a_Transaccion', 'Saldo_Despues_Transaccion']

resumen_num = val[numericas].describe().compute().T
resumen_num = resumen_num.rename(columns={'count': 'filas', 'mean': 'media', 'std': 'desv_estándar', 'min': 'mínimo',
                                          '25%': 'cuartil_1', '50%': 'mediana', '75%': 'cuartil_3', 'max': 'máximo'})
resumen_num['moda'] = [val[c].value_counts().compute().idxmax() for c in numericas]
resumen_num['asimetría'] = [float(val[c].skew().compute()) for c in numericas]
resumen_num.index.name = 'columna'
resumen_num.round(2)

,filas,media,desv_estándar,mínimo,cuartil_1,mediana,cuartil_3,máximo,moda,asimetría
columna,,,,,,,,,,
Valor,31530646.0,3057.09,1227.55,0.0,3550.0,3550.0,3550.0,3550.0,3550.0,-2.09
Saldo_Previo_a_Transaccion,31530646.0,18640.40,24624.46,0.0,5900.0,11600.0,23000.0,461000.0,0.0,4.11
Saldo_Despues_Transaccion,31530646.0,15817.88,24516.98,0.0,2850.0,8500.0,19950.0,459400.0,0.0,4.17


In [10]:
# QUÉ HACE: verifica que los valores numéricos tengan sentido (sin negativos ni saldos que suban al validar).
# CÓMO: cuenta con Dask cuántas filas cumplen cada condición "sospechosa" y calcula su porcentaje.
# QUÉ MIRAR: 0 negativos es buena señal; filas con saldo_después > saldo_previo serían recargas o errores; los ceros pueden ser transbordos o pasajes gratis (se explora más adelante).
condiciones = {
    'Valor negativo': val['Valor'] < 0,
    'Saldo previo negativo': val['Saldo_Previo_a_Transaccion'] < 0,
    'Saldo después negativo': val['Saldo_Despues_Transaccion'] < 0,
    'Saldo después MAYOR que el previo': val['Saldo_Despues_Transaccion'] > val['Saldo_Previo_a_Transaccion'],
    'Valor igual a 0': val['Valor'] == 0,
    'Saldo previo igual a 0': val['Saldo_Previo_a_Transaccion'] == 0,
}
conteos = dask.compute({nombre: cond.sum() for nombre, cond in condiciones.items()})[0]
consistencia = pd.DataFrame({'filas': conteos})
consistencia['%'] = (consistencia['filas'] / N * 100).round(3)
consistencia.index.name = 'verificación'
consistencia

,filas,%
verificación,,
Valor negativo,0,0.000
Saldo previo negativo,0,0.000
Saldo después negativo,0,0.000
Saldo después MAYOR que el previo,0,0.000
Valor igual a 0,4377978,13.885
Saldo previo igual a 0,1660336,5.266


**Lectura (sección 7, numéricas)**

- **`Valor`** solo toma **dos valores: 0 y 3,550** (la tarifa). La mediana y la moda son 3,550 y la media (3,057) queda por debajo porque el 13.9 % de las filas vale 0. La asimetría es negativa (−2.09): la "cola" va hacia los valores bajos.
- **Los saldos** son muy distintos: la media del saldo previo es 18,640 pero la mediana es 11,600, y la asimetría es positiva y alta (≈ 4.1). Hay **pocas tarjetas con saldos muy grandes** (el máximo es 461,000) que jalan el promedio hacia arriba; por eso, con los saldos, la **mediana** es una medida más representativa que la media.
- La **moda de los saldos es 0**: el 5.3 % de las validaciones entra con la tarjeta en cero.
- **Consistencia:** no hay valores negativos y el saldo después **nunca** es mayor que el previo (no hay recargas dentro de una validación). Los datos numéricos son coherentes.
- **Pendiente de entender:** las **4.4 millones de filas con `Valor` = 0** (13.9 %). No se descartan: hay que averiguar si son transbordos, pasajes gratuitos u otra cosa.

## 8. Estaciones: calidad y cruce con las validaciones

La segunda tabla trae las **coordenadas** de las estaciones, que las validaciones no tienen. Hay que revisar tres cosas:

1. **Calidad de cada archivo de estaciones:** cuántas estaciones tiene, si hay códigos repetidos, vacíos en las coordenadas o coordenadas fuera de Bogotá.
2. **Cómo se identifican las estaciones:** en las validaciones el código viene pegado al nombre (`(08000) Portal Tunal`) y en los archivos de estaciones es un número (`8000`). Hay que dejarlos en el mismo formato (5 dígitos) para poder cruzarlos.
3. **El cruce:** ¿cuántas estaciones de las validaciones tienen coordenadas y cuántas no?

In [11]:
# QUÉ HACE: deja los dos archivos de estaciones con las mismas columnas (código de 5 dígitos, nombre, latitud, longitud) y revisa su calidad.
# CÓMO: cada archivo usa nombres de columna distintos; se renombran, el código se completa con ceros a 5 dígitos y se resumen los chequeos en una tabla.
# QUÉ MIRAR: cuántas estaciones trae cada archivo; códigos repetidos; vacíos en coordenadas; que latitud y longitud caigan en Bogotá (lat ~4.4-4.9, lon ~-74.3 a -73.9).
def preparar(df, codigo, nombre, lat, lon, troncal=None):
    out = pd.DataFrame({
        'cod_estacion': df[codigo].astype(int).astype(str).str.zfill(5),
        'nombre': df[nombre].astype(str).str.strip(),
        'latitud': pd.to_numeric(df[lat], errors='coerce'),
        'longitud': pd.to_numeric(df[lon], errors='coerce'),
    })
    out['troncal'] = df[troncal] if troncal else np.nan
    return out

e1 = preparar(est_1, 'numero_estacion', 'nombre_estacion', 'latitud_estacion', 'longitud_estacion', 'troncal_estacion')
e2 = preparar(est_2, 'num_est', 'nom_est', 'latitud', 'longitud')

def chequeos(e):
    en_bogota = e['latitud'].between(4.4, 4.9) & e['longitud'].between(-74.3, -73.9)
    return {'estaciones (filas)': len(e), 'códigos distintos': e['cod_estacion'].nunique(),
            'códigos repetidos': int(e['cod_estacion'].duplicated().sum()),
            'sin latitud o longitud': int(e[['latitud', 'longitud']].isna().any(axis=1).sum()),
            'coordenadas fuera de Bogotá': int((~en_bogota & e['latitud'].notna()).sum()),
            'tiene columna troncal': 'sí' if e['troncal'].notna().any() else 'no'}

calidad_estaciones = pd.DataFrame({'archivo 1 (API)': chequeos(e1), 'archivo 2 (descarga)': chequeos(e2)})
calidad_estaciones

,archivo 1 (API),archivo 2 (descarga)
estaciones (filas),150,153
códigos distintos,150,153
códigos repetidos,0,0
sin latitud o longitud,0,0
coordenadas fuera de Bogotá,0,0
tiene columna troncal,sí,no


In [12]:
# QUÉ HACE: cruza las estaciones de las validaciones con los dos archivos de estaciones, por el código de 5 dígitos.
# CÓMO: del texto "(08000) Portal Tunal" se separa el código y el nombre; se cuenta POR CÓDIGO (un código puede tener dos nombres) y se mira en qué archivo aparece.
# QUÉ MIRAR: cuántas estaciones (y qué % de las validaciones) tienen coordenadas; el código con dos nombres; las estaciones sin coordenadas con más validaciones.
est_val = val['Estacion_Parada'].value_counts().compute().rename('validaciones').reset_index()
est_val.columns = ['Estacion_Parada', 'validaciones']
partes = est_val['Estacion_Parada'].str.extract(r'^\((\d+)\)\s*(.*)$')
est_val['cod_estacion'] = partes[0].str.zfill(5)
est_val['nombre_validaciones'] = partes[1]

nombres_por_codigo = est_val.groupby('cod_estacion')['nombre_validaciones'].nunique()
print(f"Textos de estación distintos: {len(est_val)} | códigos distintos: {est_val['cod_estacion'].nunique()}")
print('Códigos con más de un nombre:')
display(est_val[est_val['cod_estacion'].isin(nombres_por_codigo[nombres_por_codigo > 1].index)]
        [['cod_estacion', 'nombre_validaciones', 'validaciones']].reset_index(drop=True))

# una fila por código (se suman las validaciones y se conserva el nombre más frecuente)
est_val = (est_val.sort_values('validaciones', ascending=False)
                  .groupby('cod_estacion').agg(nombre_validaciones=('nombre_validaciones', 'first'),
                                               validaciones=('validaciones', 'sum')).reset_index())
est_val['en_archivo_1'] = est_val['cod_estacion'].isin(e1['cod_estacion'])
est_val['en_archivo_2'] = est_val['cod_estacion'].isin(e2['cod_estacion'])
est_val['con_coordenadas'] = est_val['en_archivo_1'] | est_val['en_archivo_2']

def categoria(f):
    if f['en_archivo_1'] and f['en_archivo_2']: return 'en los dos archivos'
    if f['en_archivo_1']: return 'solo en el archivo 1'
    if f['en_archivo_2']: return 'solo en el archivo 2'
    return 'SIN coordenadas (en ninguno)'
est_val['cruce'] = est_val.apply(categoria, axis=1)

resumen_cruce = est_val.groupby('cruce').agg(estaciones=('cod_estacion', 'count'), validaciones=('validaciones', 'sum'))
resumen_cruce['% de las validaciones'] = (resumen_cruce['validaciones'] / N * 100).round(2)
display(resumen_cruce)
print(f"Estaciones (por código) en las validaciones: {len(est_val)} | con coordenadas: {int(est_val['con_coordenadas'].sum())}")
print('\nEstaciones SIN coordenadas, de más a menos validaciones:')
sin_coordenadas = est_val[~est_val['con_coordenadas']].sort_values('validaciones', ascending=False)
sin_coordenadas[['cod_estacion', 'nombre_validaciones', 'validaciones']].head(15).reset_index(drop=True)

Textos de estación distintos: 155 | códigos distintos: 154
Códigos con más de un nombre:


,cod_estacion,nombre_validaciones,validaciones
0,09110,AV Jimenez - Inter Electricas,133040
1,09110,Avenida Jimenez,54564


,estaciones,validaciones,% de las validaciones
cruce,,,
SIN coordenadas (en ninguno),17,2600589,8.25
en los dos archivos,137,28930057,91.75


Estaciones (por código) en las validaciones: 154 | con coordenadas: 137

Estaciones SIN coordenadas, de más a menos validaciones:


,cod_estacion,nombre_validaciones,validaciones
0,50008,Corral Portal Dorado,538199
1,09127,Temporal Flores,407036
2,09126,Temporal Marly,360680
3,09125,Temporal Calle 57,313826
4,57503,Ampliacion San Mateo,303019
5,09130,Temporal Avenida 39,288277
6,09128,Temporal AV Jimenez - Inter Electricas,198928
7,04004,La Granja,75709
8,09129,Temporal Calle 22,54249
9,15002,Los Laureles,27007


**Lectura (sección 8, estaciones)**

- **Los dos archivos de estaciones son de buena calidad:** sin códigos repetidos, sin vacíos en las coordenadas y con todas las coordenadas dentro de Bogotá. El archivo 1 trae 150 estaciones **y la troncal**; el archivo 2 trae 153 y no trae troncal.
- **Las validaciones tienen 154 estaciones (por código).** Un código, el `09110`, aparece con **dos nombres** ("AV Jimenez - Inter Electricas" y "Avenida Jimenez"): se cuenta como una sola estación.
- **El cruce:** todas las estaciones que se pueden ubicar aparecen **en los dos archivos**; el archivo 2 no agrega estaciones nuevas. Con el archivo 1 (que además trae la troncal) alcanza.
- **17 estaciones no tienen coordenadas en ningún archivo y concentran el 8.25 % de las validaciones (≈ 2.6 millones).** Son estaciones **temporales** (Temporal Flores, Marly, Calle 57, Avenida 39, AV Jiménez, Calle 22), **estaciones nuevas o ampliaciones** (Ampliación San Mateo, La Granja, Los Laureles, Islandia, Tibanica) y puntos como los "Corral" (el **Corral Portal Dorado** tiene 538,199 validaciones: funciona como una estación grande) y el bicicletero y el cable de los portales.
- **Decisión:** se usa el archivo 1 como tabla de estaciones; las 17 sin coordenadas se dejan fuera del análisis geográfico (o se ubican a mano más adelante), pero **sí cuentan en el análisis de demanda**, porque son el 8 % de las validaciones.

## 9. Resumen de limpieza: qué problema hay, a cuánto afecta y qué se decide

Todo lo encontrado en las secciones anteriores, en **una sola tabla**. Los números se calculan con los resultados de las celdas anteriores (no están escritos a mano). Esta tabla es la "lista de tareas de limpieza" que se aplica al crear las tablas de análisis; el consolidado original **no se modifica**.

In [13]:
# QUÉ HACE: arma la tabla final de decisiones de limpieza con los números de las secciones anteriores y la guarda como evidencia.
# CÓMO: cada fila usa f-strings con las variables ya calculadas (faltantes, columnas, por_fecha, diferencias, consistencia, est_val...); luego se guarda en docs/decisiones_limpieza.csv.
# QUÉ MIRAR: que cada problema tenga su "a cuánto afecta" y su decisión; esta tabla va tal cual al documento.
vacias = faltantes.index[faltantes['estado'] == 'VACÍA'].tolist()
constantes = columnas.index[columnas['decisión'].str.contains('mismo valor')].tolist()
colas = por_fecha[por_fecha['estado'].str.startswith('cola')]
otro_dia = int(diferencias.loc[diferencias.index != 0, 'validaciones'].sum())
doble_nombre = nombres_por_codigo[nombres_por_codigo > 1].index.tolist()
sin_coord = est_val[~est_val['con_coordenadas']]
domingos_dia2 = frecuencias('Day_Group_Type')

decisiones = pd.DataFrame([
    ('Columnas 100 % vacías', ', '.join(vacias), '100 % de las filas',
     'No se usan al analizar (se conservan en el consolidado)'),
    ('Columnas con un solo valor', ', '.join(constantes), 'todas las filas',
     'No se usan; la hora pico se define desde la hora de la validación'),
    ('Day_Group_Type solo distingue domingos', 'Day_Group_Type', f"{domingos_dia2.loc['Dia 2', '%']} % de las filas son 'Dia 2'",
     'El tipo de día (hábil, sábado, domingo, festivo) se construye desde la fecha'),
    ('Valores faltantes en Linea', 'Linea', f"{int(faltantes.loc['Linea', 'vacíos']):,} filas ({faltantes.loc['Linea', 'vacíos_%']} %)",
     "Se dejan como 'sin línea'"),
    ('Fecha guardada como texto', 'Fecha_Transaccion', 'todas las filas',
     'Convertir a fecha-hora; extraer fecha, hora y día de la semana'),
    ('Colas de medianoche (días incompletos)', ', '.join(colas.index), f"{int(colas['validaciones'].sum()):,} filas",
     'Excluir de los promedios por tipo de día'),
    ('Validaciones en el archivo de otro día', 'Fecha_Clearing vs Fecha_Transaccion', f"{otro_dia:,} filas ({otro_dia / N * 100:.3f} %)",
     'Usar la fecha real de la validación, no la del archivo'),
    ('Código y nombre de estación pegados', 'Estacion_Parada', 'todas las filas',
     'Separar en código de 5 dígitos y nombre'),
    ('Un código con dos nombres', ', '.join(doble_nombre), f"{int(est_val.loc[est_val['cod_estacion'].isin(doble_nombre), 'validaciones'].sum()):,} filas",
     'Contar por código y usar el nombre más frecuente'),
    ('Estaciones sin coordenadas', f"{len(sin_coord)} estaciones (temporales, corrales, ampliaciones)",
     f"{int(sin_coord['validaciones'].sum()):,} filas ({sin_coord['validaciones'].sum() / N * 100:.2f} %)",
     'Se usan en la demanda; fuera del análisis geográfico (o ubicarlas a mano)'),
    ('Dos archivos de estaciones', 'estaciones_troncales.csv y _ui.csv', 'el archivo 2 no agrega estaciones',
     'Usar el archivo 1 (trae la troncal)'),
    ('Perfil y tipo de tarjeta repiten información', 'Nombre_Perfil y Tipo_Tarjeta',
     f"'Anonymous' = 'tullave Básica' ({frecuencias('Tipo_Tarjeta').loc['tullave Básica', '%']} % de las filas)",
     'Usar una sola de las dos para identificar la tarjeta anónima'),
    ('Valor = 0', 'Valor', f"{int(consistencia.loc['Valor igual a 0', 'filas']):,} filas ({consistencia.loc['Valor igual a 0', '%']} %)",
     'NO se descartan: se exploran (¿transbordos? ¿pasajes gratuitos?)'),
    ('Saldo previo = 0', 'Saldo_Previo_a_Transaccion',
     f"{int(consistencia.loc['Saldo previo igual a 0', 'filas']):,} filas ({consistencia.loc['Saldo previo igual a 0', '%']} %)",
     'NO se descartan: se exploran'),
], columns=['problema', 'dónde', 'a cuánto afecta', 'decisión'])
decisiones.index = range(1, len(decisiones) + 1)

Path('../docs').mkdir(exist_ok=True)
decisiones.to_csv('../docs/decisiones_limpieza.csv', index=False, encoding='utf-8')
pd.set_option('display.max_colwidth', 120)
decisiones

,problema,dónde,a cuánto afecta,decisión
1,Columnas 100 % vacías,"Ruta, ID_Vehiculo, Tipo_Vehiculo",100 % de las filas,No se usan al analizar (se conservan en el consolidado)
2,Columnas con un solo valor,"Fase, Hora_Pico_SN, Operador, Sistema",todas las filas,No se usan; la hora pico se define desde la hora de la validación
3,Day_Group_Type solo distingue domingos,Day_Group_Type,5.22 % de las filas son 'Dia 2',"El tipo de día (hábil, sábado, domingo, festivo) se construye desde la fecha"
4,Valores faltantes en Linea,Linea,"2,438 filas (0.008 %)",Se dejan como 'sin línea'
5,Fecha guardada como texto,Fecha_Transaccion,todas las filas,"Convertir a fecha-hora; extraer fecha, hora y día de la semana"
6,Colas de medianoche (días incompletos),"2026-08-17, 2026-09-13, 2026-09-21",325 filas,Excluir de los promedios por tipo de día
7,Validaciones en el archivo de otro día,Fecha_Clearing vs Fecha_Transaccion,"2,830 filas (0.009 %)","Usar la fecha real de la validación, no la del archivo"
8,Código y nombre de estación pegados,Estacion_Parada,todas las filas,Separar en código de 5 dígitos y nombre
9,Un código con dos nombres,09110,"187,604 filas",Contar por código y usar el nombre más frecuente
10,Estaciones sin coordenadas,"17 estaciones (temporales, corrales, ampliaciones)","2,600,589 filas (8.25 %)",Se usan en la demanda; fuera del análisis geográfico (o ubicarlas a mano)


## Qué es este conjunto de datos (resumen para alguien que lo ve por primera vez)

- **Qué es:** 31.5 millones de **validaciones** de tarjeta en las estaciones de la **troncal** de TransMilenio, durante **21 días** de agosto y septiembre de 2026 (14 días de lunes a viernes —uno de ellos festivo—, 3 sábados y 3 domingos), con **154 estaciones**.
- **Qué dice cada fila:** *cuándo* (fecha-hora al segundo), *dónde* (estación, acceso, línea, validador), *quién* (tarjeta anonimizada, perfil, tipo de tarjeta) y *cuánto* (valor y saldos).
- **Qué tan limpio está:** muy completo. Solo 15 de las 22 columnas aportan información; no hay valores negativos ni saldos que suban al validar; los problemas son de **formato** (fecha como texto, código y nombre pegados, un código con dos nombres) y de **cobertura** (17 estaciones sin coordenadas, el 8 % de las validaciones).
- **Lo que no tiene:** las validaciones solo registran **entradas** (no salidas ni cuánta gente va dentro del bus) y no traen coordenadas ni capacidades.

## Preguntas que salen de esta exploración (para elegir la pregunta analítica)

1. **¿Cómo cambia la demanda por estación, hora y tipo de día?** La demanda cambia mucho entre días hábiles, sábados, domingos y el festivo (sección 5). Falta ver *por estación y por hora*.
2. **¿Qué significan los cobros en 0 y los códigos de tarifa 2 y 3?** Son el 13.9 % de las validaciones y no parecen errores.
3. **¿Quiénes usan el sistema?** Casi 1 de cada 3 validaciones es de tarjeta anónima; el resto se reparte entre perfiles con distinta tarifa o subsidio.
4. **¿Se puede seguir a una misma tarjeta durante el día?** Hay ≈ 3.9 millones de tarjetas distintas y la columna de tarjeta está completa: se podrían ver los viajes de ida y de regreso.

Estas preguntas se profundizan en el **notebook 03** (gráficas y relaciones entre variables).